In [3]:
import xarray as xr

# 打开 nc 文件
ds = xr.open_dataset('region_hottest_month_1961_1990.nc')

# 查看变量名、坐标维度、属性
print(ds)

<xarray.Dataset> Size: 4kB
Dimensions:        (region_id: 237)
Coordinates:
  * region_id      (region_id) int64 2kB 0 1 2 3 4 5 ... 231 232 233 234 235 236
Data variables:
    hottest_month  (region_id) int64 2kB ...


In [2]:
import xarray as xr

# 打开 nc 文件
ds = xr.open_dataset('ERA5_region_t2m_1940_2024_currentyear03_to_nextyear02.nc')

# 查看变量名、坐标维度、属性
print(ds)

<xarray.Dataset> Size: 2MB
Dimensions:    (region_id: 237, year: 85, month: 12)
Coordinates:
  * region_id  (region_id) int64 2kB 0 1 2 3 4 5 6 ... 231 232 233 234 235 236
  * year       (year) int64 680B 1940 1941 1942 1943 ... 2021 2022 2023 2024
  * month      (month) int64 96B 3 4 5 6 7 8 9 10 11 12 1 2
Data variables:
    t2m        (region_id, year, month) float64 2MB ...


In [1]:
%pwd


'/home/users/kastanie/Global_regional_analysis/atmos-WRAF01-v4-1/hottestmonth'

In [1]:
import xarray as xr
import numpy as np

# 读取数据
ds_t2m = xr.open_dataset("ERA5_region_t2m_1940_2024_currentyear03_to_nextyear02.nc")
ds_hot = xr.open_dataset("region_hottest_month_1961_1990.nc")

# 提取变量
t2m = ds_t2m['t2m']  # (region_id, year, month)
hottest_month = ds_hot['hottest_month']  # (region_id,)

# 为了匹配 month 维度，需要将 hottest_month 转换成 month 维度索引（注意 month 维度从3到2）
# 构建实际月份数组
actual_months = ds_t2m['month'].values  # array([3, 4, ..., 12, 1, 2])

# 找出 hottest_month 在 month 维度中的位置索引（即 hottest_month 在 t2m 的第三维上的位置）
# 构造一个 (region_id, month) 布尔数组，标记 hottest_month 所在的 month 位置
month_idx = (actual_months.reshape(1, -1) == hottest_month.values.reshape(-1, 1)).argmax(axis=1)

# 用于 gather 的索引
# 我们要为每个 region_id 和 year 找出对应的 hottest_month index
region_ids = ds_t2m['region_id'].values
years = ds_t2m['year'].values
num_regions = len(region_ids)
num_years = len(years)

# 构造索引数组
region_idx = np.arange(num_regions).reshape(-1, 1)
year_idx = np.arange(num_years).reshape(1, -1)
month_idx_expanded = month_idx[:, np.newaxis]  # (region_id, 1)

# 提取最热月份的值 (用 xarray 的高级索引)
t2m_hottest = t2m.isel(month=xr.DataArray(month_idx_expanded, dims=["region_id", "dummy"]))

# 去掉多余维度
t2m_hottest = t2m_hottest.squeeze("dummy")  # (region_id, year)

# 创建新数据集
ds_hottest = xr.Dataset({
    't2m_hottest': t2m_hottest
})

# 保存为新 nc 文件
ds_hottest.to_netcdf("ERA5_t2m_hottest_month.nc")


In [7]:
import xarray as xr

# 打开 nc 文件
ds = xr.open_dataset('ERA5_t2m_hottest_month.nc')

# 查看变量名、坐标维度、属性
print(ds)

<xarray.Dataset> Size: 166kB
Dimensions:      (region_id: 237, year: 85)
Coordinates:
  * region_id    (region_id) int64 2kB 0 1 2 3 4 5 6 ... 231 232 233 234 235 236
  * year         (year) int64 680B 1940 1941 1942 1943 ... 2021 2022 2023 2024
    month        (region_id) int64 2kB ...
Data variables:
    t2m_hottest  (region_id, year) float64 161kB ...


In [8]:
import xarray as xr

# 打开 nc 文件
ds = xr.open_dataset('IFS_Allmonth_1981_2024_WAFregion.nc')

# 查看变量名、坐标维度、属性
print(ds)

<xarray.Dataset> Size: 13MB
Dimensions:    (year: 44, region_id: 237, number: 25, month: 12)
Coordinates:
  * year       (year) int64 352B 1981 1982 1983 1984 ... 2021 2022 2023 2024
  * region_id  (region_id) int64 2kB 0 1 2 3 4 5 6 ... 231 232 233 234 235 236
  * number     (number) int64 200B 0 1 2 3 4 5 6 7 8 ... 17 18 19 20 21 22 23 24
  * month      (month) int64 96B 3 4 5 6 7 8 9 10 11 12 1 2
Data variables:
    t2m        (year, region_id, number, month) float32 13MB ...


In [2]:
import xarray as xr
import numpy as np

# 读取数据
ds_ifs = xr.open_dataset("IFS_Allmonth_1981_2024_WAFregion.nc")
ds_hot = xr.open_dataset("region_hottest_month_1961_1990.nc")

t2m = ds_ifs['t2m']  # (year, region_id, number, month)
hottest_month = ds_hot['hottest_month']  # (region_id,)

# 实际 month 维度对应的月份顺序，比如 [3,4,...,12,1,2]
actual_months = ds_ifs['month'].values

# 获取每个区域最热月份对应 month 维度的索引
month_idx = np.array([np.where(actual_months == m)[0][0] for m in hottest_month.values])

# 维度大小
num_years = ds_ifs.sizes['year']
num_regions = ds_ifs.sizes['region_id']
num_members = ds_ifs.sizes['number']

# 准备空数组存储结果 (year, region_id, number)
result = np.full((num_years, num_regions, num_members), np.nan, dtype=np.float32)

# 按 region_id 遍历，取对应最热月数据
for rid in range(num_regions):
    idx = month_idx[rid]
    # 取所有年份、该区域所有成员在该月的 t2m
    result[:, rid, :] = t2m[:, rid, :, idx].values

# 构造新的 DataArray
da_result = xr.DataArray(
    result,
    dims=['year', 'region_id', 'number'],
    coords={
        'year': ds_ifs['year'],
        'region_id': ds_ifs['region_id'],
        'number': ds_ifs['number']
    },
    name='t2m_hottest_month'
)

# 保存成新的 NetCDF 文件
da_result.to_netcdf("IFS_t2m_hottest_month.nc")


In [3]:
import xarray as xr

# 打开 nc 文件
ds = xr.open_dataset('IFS_t2m_hottest_month.nc')

# 查看变量名、坐标维度、属性
print(ds)

<xarray.Dataset> Size: 1MB
Dimensions:            (year: 44, region_id: 237, number: 25)
Coordinates:
  * year               (year) int64 352B 1981 1982 1983 1984 ... 2022 2023 2024
  * region_id          (region_id) int64 2kB 0 1 2 3 4 5 ... 232 233 234 235 236
  * number             (number) int64 200B 0 1 2 3 4 5 6 ... 19 20 21 22 23 24
Data variables:
    t2m_hottest_month  (year, region_id, number) float32 1MB ...


In [1]:
import xarray as xr
import scipy.signal

# 打开数据
ds = xr.open_dataset("IFS_t2m_hottest_month.nc")
t2m = ds['t2m_hottest_month']  # (year, region_id, number)

# 仅选择1981–2023年的数据
t2m_subset = t2m.sel(year=slice(1981, 2024))

# 去趋势函数
def detrend_1d(arr):
    return scipy.signal.detrend(arr, type='linear')

# 使用 apply_ufunc 沿 year 维度去趋势
t2m_detrended = xr.apply_ufunc(
    detrend_1d,
    t2m_subset,
    input_core_dims=[['year']],
    output_core_dims=[['year']],
    vectorize=True,
    dask='allowed',
    output_dtypes=[t2m.dtype]
)

# 保存新数据
ds_out = xr.Dataset({'t2m_hottest_month_detrended': t2m_detrended})
ds_out.to_netcdf("IFS_t2m_hottest_month_detrended_1981_2024.nc")
print("1981–2024年去趋势数据已保存")


1981–2024年去趋势数据已保存


In [6]:
import xarray as xr

# 打开 nc 文件
ds = xr.open_dataset('IFS_t2m_hottest_month_detrended_1981_2023.nc')

# 查看变量名、坐标维度、属性
print(ds)

<xarray.Dataset> Size: 1MB
Dimensions:                      (year: 43, region_id: 237, number: 25)
Coordinates:
  * year                         (year) int64 344B 1981 1982 1983 ... 2022 2023
  * region_id                    (region_id) int64 2kB 0 1 2 3 ... 234 235 236
  * number                       (number) int64 200B 0 1 2 3 4 ... 21 22 23 24
Data variables:
    t2m_hottest_month_detrended  (region_id, number, year) float32 1MB ...


In [1]:
import xarray as xr

# 打开 nc 文件
ds = xr.open_dataset('grid_comparison_fidelity.nc')

# 查看变量名、坐标维度、属性
print(ds)

<xarray.Dataset> Size: 523kB
Dimensions:         (latitude: 180, longitude: 360)
Coordinates:
  * latitude        (latitude) float64 1kB -89.5 -88.5 -87.5 ... 87.5 88.5 89.5
  * longitude       (longitude) float64 3kB -179.5 -178.5 -177.5 ... 178.5 179.5
Data variables:
    fidelity_score  (latitude, longitude) float64 518kB ...
